# Data Preprocessing
This notebook starts with downloading the dataset and then performs data preprocessing steps to create a clean dataset for further analysis.

## Imports

In [1]:
import os
import zipfile
import requests
from tqdm import tqdm
import shutil
import csv
import re
import json
from collections import Counter, defaultdict
import random
import pandas as pd

## Downloading the Dataset and extracting it

This ran more than 30 min even with really fast internet I think they are throttling the download speed.

In [ ]:
output_dir = "data/dta_complete_2021-10-19"
os.makedirs(output_dir, exist_ok=True)
url = "https://www.deutschestextarchiv.de/media/download/dta_komplett_2021-10-19_tcf.zip"

# Download
with requests.get(url, stream=True) as r, open("data/dta.zip", "wb") as f:
    total_size = int(r.headers.get('content-length', 0))
    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Downloading") as t:
        for chunk in r.iter_content(1024):
            f.write(chunk)
            t.update(len(chunk))
print("\nDownload completed. Extracting files...")

# Extract
with zipfile.ZipFile("data/dta.zip", "r") as zip_ref:
    zip_ref.extractall(output_dir)
print("Extraction completed. Processing files...")

# Cleanup and rearrange
shutil.rmtree(os.path.join(output_dir, "dta_komplett_2021-10-19", "simple"), ignore_errors=True)
full_path = os.path.join(output_dir, "dta_komplett_2021-10-19", "full")
for file in os.listdir(full_path):
    shutil.move(os.path.join(full_path, file), output_dir)
print("Files processed and moved to the output directory.")

# Remove leftover dirs
shutil.rmtree(os.path.join(output_dir, "dta_komplett_2021-10-19"), ignore_errors=True)
os.remove("data/dta.zip")
print("Cleanup completed. The dataset is ready for use in the output directory:", output_dir)

## Take only 0.1% of the dataset for testing

In [4]:
output_dir = "data/dta_complete_2021-10-19"
#create dir with 0.1% of the dataset randomly sampled
test_dir = "data_example/dta_0.1_percent"
os.makedirs(test_dir, exist_ok=True)
# Get all files in the output directory
files = [f for f in os.listdir(output_dir) if os.path.isfile(os.path.join(output_dir, f))]
# Sample 0.1% of the files
sample_size = max(1, len(files) // 1000)  # Ensure at least one file is selected
sampled_files = random.sample(files, sample_size)
# Move sampled files to the test directory
for file in sampled_files:
    shutil.copy(os.path.join(output_dir, file), os.path.join(test_dir, file))
print(f"Sampled {sample_size} files for testing in the directory: {test_dir}")

one_percent = True

Sampled 4 files for testing in the directory: data_example/dta_0.1_percent


## Functions for Data Preprocessing

In [6]:
def extract_sentences_from_text(text):
    """
    Extrahiere rohe XML-Fragmente, die aus <token>…</token>-Elementen bestehen
    und mit einem Punkt-Token enden.
    """
    # Pattern für ein ganzes Satzfragment (eine Reihe von Token-Elementen, endend auf Punkt)
    sentence_re = re.compile(
        r'(?:<token\b[^>]*>.*?</token>\s*)+?<token\b[^>]*>\.</token>',
        flags=re.DOTALL | re.IGNORECASE
    )
    return [m.group(0).strip() for m in sentence_re.finditer(text)]

def process_files(input_dir, keyword_level, examples_per_keyword):
    """
    Durchsuche alle .xml/.tcf, match Keywords in den rohen XML-Sätzen,
    sammle bis zu examples_per_keyword Beispiele und zähle Frequenzen.
    """
    examples = defaultdict(list)  # kw -> list of dicts {raw_xml, file, sentence_id}
    freq     = Counter()
    # Precompile Regex für Keywords (aber weiter über raw text matchen)
    patterns = {
        kw: re.compile(rf'\b{re.escape(kw)}\b', flags=re.IGNORECASE)
        for kw in keyword_level
    }

    # Zähle alle passenden Dateien für tqdm
    file_list = []
    for root, _, files in os.walk(input_dir):
        for fname in files:
            if fname.lower().endswith(('.xml', '.tcf')):
                file_list.append((root, fname))

    for root, fname in tqdm(file_list, desc="Dateien werden verarbeitet"):
        path = os.path.join(root, fname)
        try:
            raw = open(path, encoding='utf-8').read()
        except Exception as e:
            print(f"Warning: {fname}: {e}")
            continue

        sentences = extract_sentences_from_text(raw)
        for sid, raw_xml in enumerate(tqdm(sentences, desc=f"Sätze in {fname}", leave=False)):
            for kw, pat in patterns.items():
                if pat.search(raw_xml):
                    freq[kw] += 1
                    if len(examples[kw]) < examples_per_keyword:
                        examples[kw].append({
                            "raw_xml": raw_xml,
                            "file": fname,
                            "sentence_id": sid
                        })
            # Abbruch, wenn alle Limits erreicht
            if all(len(examples[k]) >= examples_per_keyword for k in keyword_level):
                return examples, freq

    return examples, freq

def write_output_json(output_path, examples, freq, keyword_level):
    """
    Schreibe ein JSON-Array mit allen Matches:
    [
      {
        "file": "...",
        "sentence_id": 42,
        "raw_xml": "<token ...>…</token>…",
        "keyword": "je",
        "score": 4,
        "frequency": 21
      },
      …
    ]
    """
    out = []
    for kw, hits in tqdm(examples.items(), desc="Schreibe Output-JSON"):
        score = keyword_level.get(kw, "")
        count = freq.get(kw, 0)
        for hit in hits:
            out.append({
                "file": hit["file"],
                "sentence_id": hit["sentence_id"],
                "raw_xml": hit["raw_xml"],
                "keyword": kw,
                "score": score,
                "frequency": count
            })

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON-Output geschrieben nach {output_path}")

def load_testset(testset_path):
    """
    Load keywords and their scores from a CSV with columns 'Keyword','Score'.

    Returns:
        dict: mapping keyword -> score
    """
    keyword_level = {}
    with open(testset_path, encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in tqdm(reader, desc="Lade Testset"):
            kw = row.get("Keyword", "").strip()
            score = row.get("Score", "").strip()
            if kw:
                keyword_level[kw] = score
    return keyword_level


## Running the Preprocessing

In [8]:
if one_percent:
    input_dir = test_dir
    output_path_json     = "data_example/annotated_data_0.1_percent.json"
    print(f"Using test directory: {input_dir}")
    output_path_json = "data_example/annotated_data_0.1_percent.json"
else:
    input_dir = output_dir
    print(f"Using full dataset directory: {input_dir}")
    output_path_json = "data/annotated_data.json"

testset_path         = "testset.csv"

examples_per_keyword = 2

keyword_level = load_testset(testset_path)
examples, freq = process_files(input_dir, keyword_level, examples_per_keyword)
write_output_json(output_path_json, examples, freq, keyword_level)

Using test directory: data_example/dta_0.1_percent


Lade Testset: 235it [00:00, 397604.45it/s]
Schreibe Output-JSON: 100%|██████████| 37/37 [00:00<00:00, 838860.80it/s]

✅ JSON-Output geschrieben nach data_example/annotated_data_0.1_percent.json


## Analysis of the Dataset

In [9]:
# Assuming freq_no_date is a Counter for the non-dated data
keyword_stats = []
keywords = list(keyword_level.keys())
for kw in keywords:
    count_label = freq.get(kw, 0)
    count_no_date = freq.get(kw, 0)
    not_in_dataset = int(count_label == 0 and count_no_date == 0)
    keyword_stats.append({
        "Keyword": kw,
        "Label_Count": count_label,
    })
if one_percent:
    output_path_csv = "data_example/keyword_stats_0.1_percent.csv"
else:
    output_path_csv = "data/keyword_stats.csv"

df_stats = pd.DataFrame(keyword_stats)
df_stats.to_csv(output_path_csv, index=False)
print(f"Saved keyword statistics to {output_path_csv}")


Saved keyword statistics to data_example/keyword_stats_0.1_percent.csv
